## Wolfram Alpha tool
Allows our AI agent to access Wolfram Alpha’s computational engine to solve complex math, physics, and factual queries.
### Use Cases
- **Mathematical Solving**: Calculating integrals, solving differential equations, or performing matrix manipulations.
- **Scientific Queries**: Retrieving physical constants, chemical properties, or astronomical data.
- **Factual Data**: Accessing real-time demographics, economic indicators, or unit conversions. 

Instructions: https://docs.langchain.com/oss/python/integrations/tools/wolfram_alpha

In [31]:
!uv add wolframalpha

Resolved 224 packages in 2ms
Audited 213 packages in 7ms


In [ ]:
import os
from dotenv import load_dotenv
from typing import List, Any, Optional, Dict
from sidekick import State
from langchain_community.utilities.wolfram_alpha import WolframAlphaAPIWrapper
from langchain_community.tools.wolfram_alpha.tool import WolframAlphaQueryRun

load_dotenv(override=True)

wolfram_token = os.environ["WOLFRAM_ALPHA_APPID"]


In [ ]:
async def wolfram_tool():
    wolfram = WolframAlphaAPIWrapper()
    wolfram_tool = WolframAlphaQueryRun(api_wrapper=wolfram)
    return wolfram_tool

wolfram = await wolfram_tool()
print(wolfram.run("What is 2x+5 = -3x + 7?"))

def wolfram_knowledge_worker(state: State) -> Dict[str, Any]:
    # Get the user's query
    query = state["input"]

    # Use the Wolfram Alpha tool to get the answer
    answer = wolfram.run(query)
    return {"answer": [answer]}

Assumption: 2 x + 5 = -3 x + 7 
Answer: x = 2/5


## Planning Agent
This agent will plan the sequence of events when tackling the user's query.
It will throw back 3 questions to the user to further seek clarification then formulate the final query that it will delegate to worker agent and the scientific agent depending on the nature of the question.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing_extensions import Literal
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from sidekick import State, Sidekick

class MySidekick(Sidekick):
    def __init__(self):
        super().__init__()
        planner_llm = ChatOpenAI(model="gpt-4.1-mini")
        planner_llm_with_tools = planner_llm.bind_tools(self.tools)

    def planner(self, state: State) -> State:
        system_message = f"""You are a helpful assistant that receives the user's query and plans the sequence of events 
        to tackle the query.
        You will first throw back 3 questions to the user to further seek clarification. The questions should be based 
        on the user's query and should be designed to help you understand the user's query better.
        After, you will then formulate the final query that it will delegate to worker agent or the scientific agent 
        depending on the nature of the question.
        Route the input to general, scientific, or math based on the user's query.

        This is the success criteria:
        {state["success_criteria"]}

        Once you are finished, reply with the final answer, and don't ask a question; simply reply with the answer.
        """

        if state.get("feedback_on_work"):
            system_message += f"""
                Previously you thought you completed the assignment, but your reply was rejected because the success criteria was not met.
                Here is the feedback on why this was rejected:
                {state["feedback_on_work"]}
                With this feedback, please continue the assignment, ensuring that you meet the success criteria or have a question for the user.
                """
        
        # Add in the system message
        found_system_message = False
        messages = state["messages"]
        for message in messages:
            if isinstance(message, SystemMessage):
                message.content = system_message
                found_system_message = True

        if not found_system_message:
            messages = [SystemMessage(content=system_message)] + messages

        # Augment the LLM with schema for structured output
        router = self.llm_with_tools.with_structured_output(Route)

        # Run the augmented LLM with structured output to serve as routing logic
        decision = router.invoke(
            [
                SystemMessage(content="Route the input to general, scientific, or math based on the user's request."),
                HumanMessage(content=messages),
            ]
        )

        return {"decision": decision.step}

# Schema for structured output to use as routing logic
class Route(BaseModel):
    step: Literal["general", "scientific", "math"] = Field(
        None, description="The next step in the routing process"
    )



In [ ]:
from aiohttp import worker


my_sidekick = MySidekick()

planner = my_sidekick.planner()
general_worker = my_sidekick.worker()
evaluator = my_sidekick.evaluator()
wolfram_worker = wolfram_knowledge_worker()

In [ ]:
# Conditional edge function to route to the appropriate node
def route_decision(state: State):
    # Return the node name you want to visit next
    if state["decision"] == "scientific":
        return "scientific"
    elif state["decision"] == "math":
        return "math"
    elif state["decision"] == "general":
        return "general"
    else:
        return "evaluator"

In [ ]:

# Build workflow
router_builder = StateGraph(State)

# Add nodes
router_builder.add_node("planner", planner)
router_builder.add_node("general", worker)
router_builder.add_node("scientific", wolfram_worker)
router_builder.add_node("math", wolfram_worker)
router_builder.add_node("evaluator", evaluator)

# Add edges to the nodes
router_builder.add_edge(START, "planner")

# Add conditional edges
router_builder.add_conditional_edges(
    "planner", route_decision, 
    {"general": "general", "scientific": "scientific", "math": "math", "evaluator": "evaluator"}
)
router_builder.add_edge("general", "planner")
router_builder.add_edge("scientific", "planner")
router_builder.add_edge("math", "planner")
router_builder.add_edge("evaluator", "planner")

router_builder.add_edge("planner", END)

# Compile workflow
router_workflow = router_builder.compile()


## Gradio UI

In [ ]:
import gradio as gr


async def setup():
    sidekick = MySidekick()
    await sidekick.setup()
    return sidekick


async def process_message(sidekick, message, success_criteria, history):
    results = await sidekick.run_superstep(message, success_criteria, history)
    return results, sidekick


async def reset():
    new_sidekick = MySidekick()
    await new_sidekick.setup()
    return "", "", None, new_sidekick


def free_resources(sidekick):
    print("Cleaning up")
    try:
        if sidekick:
            sidekick.cleanup()
    except Exception as e:
        print(f"Exception during cleanup: {e}")


with gr.Blocks(title="Sidekick", theme=gr.themes.Default(primary_hue="emerald")) as ui:
    gr.Markdown("## Sidekick Personal Co-Worker")
    sidekick = gr.State(delete_callback=free_resources)

    with gr.Row():
        chatbot = gr.Chatbot(label="Sidekick", height=300, type="messages")
    with gr.Group():
        with gr.Row():
            message = gr.Textbox(show_label=False, placeholder="Your request to the Sidekick")
        with gr.Row():
            success_criteria = gr.Textbox(
                show_label=False, placeholder="What are your success critiera?"
            )
    with gr.Row():
        reset_button = gr.Button("Reset", variant="stop")
        go_button = gr.Button("Go!", variant="primary")

    ui.load(setup, [], [sidekick])
    message.submit(
        process_message, [sidekick, message, success_criteria, chatbot], [chatbot, sidekick]
    )
    success_criteria.submit(
        process_message, [sidekick, message, success_criteria, chatbot], [chatbot, sidekick]
    )
    go_button.click(
        process_message, [sidekick, message, success_criteria, chatbot], [chatbot, sidekick]
    )
    reset_button.click(reset, [], [message, success_criteria, chatbot, sidekick])


ui.launch(inbrowser=True)